
# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: `grid.update()` is called once, flushing all hex style mutations into the `hexes` layer.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay()` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay()` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width)` | 60 | SVG | River network from drainage basins |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | Overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'}
# If basins=None, logs: "Skipping rivers: missing {'basins'}"
```

---

## Creating Custom Overlays

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Custom overlay with hex style mutations

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        # ctx.basins is guaranteed present because of requires
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Create custom | Write a function returning `OverlaySpec` |
```

In [ ]:
#| default_exp overlay

In [ ]:
#| export
#| hide
#import nbdev; nbdev.nbdev_export()
import sys
import math
from fastcore.basics import patch

#| export
## Getting Started

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex


from HexMagic.styles import StyleCSS,  SVGBuilder, SVGLayer

In [ ]:
#| export
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain


Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, SoilSystem


In [ ]:
#| export
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.climate import TerrainFactory

In [ ]:
#| export

from HexMagic.terraform import Terraform
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.core import TerraDemo
from HexMagic.primitives import HexTouchMap
from HexMagic.water.soil import SoilSystem, SoilType


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass

In [ ]:
#!cat ../docs/ll*

In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

Can you start to build TerrainDisplay

In [ ]:
#| export
#| export
import logging
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class OverlaySpec:
    """Specification for a single overlay layer."""
    name: str
    renderer: Callable[['OverlayContext'], str]
    requires: set = field(default_factory=set)
    priority: int = 50



In [ ]:
#| export
@dataclass
class OverlayContext:
    """Rendering context passed to overlay renderers."""
    terrain: 'Terrain'
    grid: 'HexGrid'
    builder: 'SVGBuilder'
    extras: dict = field(default_factory=dict)

    def __getattr__(self, name):
        if name in self.__dict__.get('extras', {}):
            return self.extras[name]
        raise AttributeError(f"OverlayContext has no '{name}' — did you forget to pass it?")


In [ ]:
#| export
#| export
def TerrainDisplay(
    *overlays: OverlaySpec,
    terrain: 'Terrain',
    result=None, board=None, c2f=None,
    basins=None, soil=None,
    radius=None, debug=False,
    **attrs
):
    grid = terrain.hexGrid
    if radius: grid.adjustRadius(radius)

    if basins is None and result and hasattr(result, 'basins'):
        basins = result.basins
    if soil is None and result and hasattr(result, 'soil'):
        soil = result.soil

    ctx = OverlayContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        extras=dict(result=result, board=board, c2f=c2f,
                    basins=basins, soil=soil)
    )

    available = {'terrain', 'grid', 'builder'}
    if soil:   available.add('soil')
    if result: available.add('result')
    if board:  available.add('board')
    if c2f:    available.add('c2f')
    if basins: available.add('basins')

    # Phase 1: Run renderers, collect SVG strings (+ hex style side effects)
    collected = []
    for spec in sorted(overlays, key=lambda o: o.priority):
        missing = spec.requires - available
        if missing:
            logging.warning(f"Skipping {spec.name}: missing {missing}")
            continue
        try:
            svg = spec.renderer(ctx)
            if svg: collected.append((spec.name, svg))
        except Exception as e:
            logging.error(f"Overlay {spec.name} failed: {e}")

    # Phase 2: Bake hex style mutations into the hexes layer
    grid.update()

    # Phase 3: Add SVG overlay layers ON TOP of hexes
    for name, svg in collected:
        grid.builder.adjust(name, svg)

    if debug: return grid.builder.__ft__()

    return Div(
        HexTouchMap(grid, cls=attrs.pop('map_cls', 'w-full h-full')),
        **attrs
    )


In [ ]:
myTerr = TerraDemo().japan_korea_map()

In [ ]:
myTerr.compute_climate()

In [ ]:
myTerr.carve_to_ocean(num_lakes=0)
basins = DrainageBasins(myTerr)

In [ ]:
#| export
#| export
def TerrainOverlay(**kw) -> OverlaySpec:
    """Color hexes by elevation using terrain color levels."""
    def render(ctx: OverlayContext) -> str:
        ctx.terrain.colorMap()
        return ""  # Pure hex style mutation — grid.update() bakes it in
    return OverlaySpec("heights", render, priority=5)


In [ ]:
#| export
#| export
def ClimateOverlay(levels: int = 5, min_density: float = 0.25, **kw) -> OverlaySpec:
    """Climate zone dots colored by climate type, density by precipitation."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.dottedClimate(flow_levels=levels, min_density=min_density)
    return OverlaySpec("climate", render, priority=40)



def WatershedOverlay(**kw) -> OverlaySpec:
    """Watershed boundary dot overlay."""
    def render(ctx: OverlayContext) -> str:
        if not ctx.result or not hasattr(ctx.result, 'basins') or not ctx.result.basins:
            return ""
        return ctx.result.basins.dotted_watershed_overlay()
    return OverlaySpec("watersheds", render, requires={'result'}, priority=35)


In [ ]:
#| export
def RiverOverlay(top_n: int = 8, simplify_k: int = 3, max_width: float = None, lake_base_size: int = 3, **kw) -> OverlaySpec:
    """River network from drainage basins."""
    def render(ctx: OverlayContext) -> str:
        if not ctx.basins: return ""
        return ctx.basins.draw_watersheds(
            top_n=top_n, simplify_k=simplify_k,
            max_width=max_width, lake_base_size=lake_base_size
        )
    return OverlaySpec("rivers", render, requires={'basins'}, priority=60)


TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

In [ ]:
myTerr.hexGrid.builder.layers = []
myTerr.hexGrid.builder.layers = []

In [ ]:
#| export
#| export
def CreamOverlay(**kw) -> OverlaySpec:
    """Parchment-style base fills using elevation + climate + coast distance."""
    def render(ctx: OverlayContext) -> str:
        terrain, grid, builder = ctx.terrain, ctx.grid, ctx.builder

        terrain_fills = {
            'ocean':     '#E3F2FD',
            'coast':     '#F8F4E8',
            'lowland':   '#FDF5E6',
            'plains':    '#FAF0D4',
            'hills':     '#EFE6D5',
            'highlands': '#E8DFD0',
            'mountain':  '#DED4C4',
        }

        styles = {k: StyleCSS(k, fill=v) for k, v in terrain_fills.items()}
        for s in styles.values():
            builder.add_style(s)

        if 'distance_to_coast' not in terrain.fields:
            terrain.compute_distance_to_coast()

        for i in range(len(terrain.elevations)):
            elev = terrain.elevations[i]
            dist_coast = terrain.fields['distance_to_coast'][i]

            if elev <= 0:                              style = styles['ocean']
            elif dist_coast <= 2 and elev < 200:       style = styles['coast']
            elif elev < 200:                           style = styles['lowland']
            elif elev < 500:                           style = styles['plains']
            elif elev < 1200:                          style = styles['hills']
            elif elev < 2000:                          style = styles['highlands']
            else:                                      style = styles['mountain']

            grid.hexes[i].style = style

        return ""  # Pure hex style mutation — grid.update() bakes it in
    return OverlaySpec("cream", render, priority=10)


In [ ]:
#| export
#| export
def SoilOverlay(f=None, **kw) -> OverlaySpec:
    """Soil type hatch pattern overlay showing bedrock types."""
    def render(ctx: OverlayContext) -> str:
        soil = getattr(ctx, 'soil', None)
        if soil is None:
            soil = SoilSystem.from_plates(ctx.terrain, [])

        terrain, grid, builder = ctx.terrain, ctx.grid, ctx.builder
        patGen = TerrainPatterns(terrain)
        cols = [x.to_nc() for x in SoilType.standard_types()]

        # Ocean wave pattern
        ocean_hexes = terrain.find_region_at_level(0)
        ocean_region = HexRegion(hexes=ocean_hexes, hexGrid=grid)

        wave = patGen.wavePattern("ocean_waves_pat",
                                  amplitude=4, wavelength=16,
                                  color="#1565C0", fill="#E3F2FD")
        oceanStyle = StyleCSS("ocean", fill="url(#ocean_waves_pat)")
        builder.add_definition(wave)
        builder.add_style(oceanStyle)

        # Soil hatch patterns
        patterns, soilStyles = patGen.namedHatchPattern(cols, stroke_width=4, spacing=8)
        for p in patterns: builder.add_definition(p)
        for s in soilStyles: builder.add_style(s)

        # Set hex styles by soil region
        for i, region in enumerate(soil.regions):
            style = soilStyles[i]
            for h in region:
                grid.hexes[h].style = style

        for i in ocean_region:
            grid.hexes[i].style = oceanStyle

        return ""  # Pure hex style mutation — grid.update() bakes it in
    return OverlaySpec("soil", render, priority=15)


In [ ]:
#| export
#| export
def FlowOverlay(**kw) -> OverlaySpec:
    """Flow direction arrows showing drainage patterns."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.flow_diagram()
    return OverlaySpec("flow", render, priority=42)


In [ ]:
#| export
def TemperatureOverlay(ocean_color: str = "#2c5f8d", **kw) -> OverlaySpec:
    """Temperature gradient overlay — purple→blue→green→yellow→red."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.render_icon_temperature(ocean_color=ocean_color)
    return OverlaySpec("temperature", render, priority=41)


In [ ]:
#| export
#| export
def PrecipitationOverlay(levels: int = 5, min_density: float = 0.25, 
                          color: str = "#4A90E2", **kw) -> OverlaySpec:
    """Precipitation density dots — denser dots = more rainfall."""
    def render(ctx: OverlayContext) -> str:
        if 'precipitation' not in ctx.terrain.fields:
            ctx.terrain.compute_weather()
        return ctx.terrain.dotted_field(
            field='precipitation',
            flow_levels=levels,
            min_density=min_density,
            color=color
        )
    return OverlaySpec("precipitation", render, priority=43)


In [ ]:
soil = SoilSystem.from_plates(myTerr, [])

In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    
    CreamOverlay(),
    SoilOverlay(),
    RiverOverlay(),
    soil=soil,       # ← pass it here

    
    basins=basins,
    terrain=myTerr,
    
    debug=True
)


I think this is all so strong. Can you do a design doc on how to use this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```markdown
# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: `grid.update()` is called once, flushing all hex style mutations into the `hexes` layer.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay()` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay()` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width)` | 60 | SVG | River network from drainage basins |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | Overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'}
# If basins=None, logs: "Skipping rivers: missing {'basins'}"
```

---

## Creating Custom Overlays

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Custom overlay with hex style mutations

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        # ctx.basins is guaranteed present because of requires
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Create custom | Write a function returning `OverlaySpec` |
```

In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    
    TerrainOverlay(),
    RiverOverlay(),
    basins=basins,
    terrain=myTerr,
    
    debug=False
)